In [7]:
import pandas as pd
import numpy as np
import spacy
from textblob import TextBlob
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import chi2
from tqdm.auto import tqdm

In [12]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA = BASE_DIR / "data/processed/news_with_features.csv"

In [14]:
df = pd.read_csv(DATA)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/data/processed/news_with_features.csv'

In [ ]:
#Now let's add simple, cheap features to the data
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [9]:
def get_tech_list(val):
    try:
        items = json.loads(val.replace("'", '"')) if isinstance(val, str) else val
        return [i['technique'] for i in items]
    except: return []

def extract_features(text):
    if not isinstance(text, str) or not text.strip():
        return [0] * 13

    blob = TextBlob(text)
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if not t.is_punct]
    word_cnt = max(len(words), 1)

    return [
        len(text), len(words),
        np.mean([len(w.text) for w in words]) if words else 0,
        sum(1 for w in words if w.text.isupper() and len(w.text) > 1) / word_cnt,
        sum(1 for t in tokens if any(c in '!?."' for c in t.text)) / word_cnt,
        len(doc), sum(1 for w in words if w.text.istitle()), len(tokens),
        blob.sentiment.polarity, blob.sentiment.subjectivity,
        sum(1 for t in doc if t.pos_ == "ADJ") / len(tokens),
        sum(1 for t in doc if t.pos_ == "ADV") / len(tokens),
        len(set([w.text.lower() for w in words])) / word_cnt
    ]

In [ ]:
#Process Features
feature_cols = ['char_count', 'word_count', 'avg_word_len', 'caps_ratio', 'punct_ratio',
                'sent_count', 'title_count', 'total_tokens', 'polarity',
                'subjectivity', 'adj_density', 'adv_density', 'ttr']

In [10]:
handcrafted_features = np.array(df['text'].apply(extract_features).tolist())
features_df = pd.DataFrame(handcrafted_features, columns=feature_cols, index=df.index)
df = pd.concat([df, features_df], axis=1)

Getting features for all articles (this will take a while)...


NameError: name 'news' is not defined

In [ ]:
#Vectorize and split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['text'].fillna(""))
X_test_tfidf = tfidf.transform(test_df['text'].fillna(""))

In [ ]:
#Enhance with keywords
all_techs = sorted(list(set([t for s in train_df['propaganda'].apply(get_tech_list) for t in s])))
mlb = MultiLabelBinarizer(classes=all_techs)

train_has_p = train_df['propaganda'].apply(get_tech_list).str.len() > 0
y_train_spec_full = mlb.fit_transform(train_df['propaganda'].apply(get_tech_list))

In [ ]:
keyword_feats_train = []
keyword_feats_test = []

for i, tech in enumerate(all_techs):
    _, pval = chi2(X_train_tfidf, y_train_spec_full[:, i])
    top_indices = np.argsort(pval)[:15]

    #Create binary feature: does text contain any of these top 15 words?
    keyword_feats_train.append(X_train_tfidf[:, top_indices].sum(axis=1) > 0)
    keyword_feats_test.append(X_test_tfidf[:, top_indices].sum(axis=1) > 0)

In [ ]:
X_train_final = np.hstack([train_df[feature_cols].values, X_train_tfidf.toarray(), np.array(keyword_feats_train).T.reshape(len(train_df), -1)])
X_test_final = np.hstack([test_df[feature_cols].values, X_test_tfidf.toarray(), np.array(keyword_feats_test).T.reshape(len(test_df), -1)])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

In [ ]:
#Train lightweight version of scanner, or gatekeeper model
y_train_bin = train_has_p.astype(int)
y_test_bin = (test_df['propaganda'].apply(get_tech_list).str.len() > 0).astype(int)

gatekeeper = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
gatekeeper.fit(X_train_scaled, y_train_bin)
print(f"Gatekeeper F1: {f1_score(y_test_bin, gatekeeper.predict(X_test_scaled)):.4f}")

In [ ]:
#Train lightweight version of classifier model
X_train_spec = X_train_scaled[train_has_p]
y_train_spec = y_train_spec_full[train_has_p]

test_has_p = y_test_bin == 1
X_test_spec = X_test_scaled[test_has_p]
y_test_spec = mlb.transform(test_df[test_has_p]['propaganda'].apply(get_tech_list))

spec_model = MultiOutputClassifier(LogisticRegression(class_weight='balanced', max_iter=2000))
spec_model.fit(X_train_spec, y_train_spec)

In [ ]:
#Optimize thresholds
probs = spec_model.predict_proba(X_test_spec)
y_pred_opt = np.zeros(y_test_spec.shape)

for i, tech in enumerate(all_techs):
    p, r, thresholds = precision_recall_curve(y_test_spec[:, i], probs[i][:, 1])
    f1 = 2 * (p * r) / (p + r + 1e-10)
    best_thresh = thresholds[np.argmax(f1)]
    y_pred_opt[:, i] = (probs[i][:, 1] >= best_thresh).astype(int)

In [ ]:
print("\nFinal Specialist Report:")
print(classification_report(y_test_spec, y_pred_opt, target_names=all_techs, zero_division=0))